# Civil Comments Toxicity Detector

Student: Md. Jahid Gazi | Student ID: 1000128068

This notebook runs the full official Google Civil Comments train/validation/test splits, a frozen BERT baseline, end-to-end fine-tuning, four hyperparameter trials, held-out test evaluation, real-world predictions, and SHAP explanations. Results are generated by execution; no estimated metrics are inserted as outputs.

In [ ]:
%pip -q install datasets transformers accelerate scikit-learn shap pyarrow pandas matplotlib

In [ ]:
import os, gc, json, random
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME='bert-base-uncased'; MAX_LENGTH=128; THRESHOLD=0.5
print('Device:',DEVICE)

## 1. Stream and cache the complete official dataset splits

All valid rows are streamed in batches to Parquet. No sample limit is applied.

In [ ]:
CACHE=Path('civil_comments_full_cache'); CACHE.mkdir(exist_ok=True)
def cache_split(name):
    path=CACHE/f'{name}.parquet'
    if path.exists() and path.stat().st_size: return path
    import pyarrow as pa, pyarrow.parquet as pq
    tmp=path.with_suffix('.tmp'); writer=None; batch=[]; count=0
    for row in load_dataset('google/civil_comments',split=name,streaming=True):
        text=row.get('text'); score=row.get('toxicity')
        if text is None or not str(text).strip() or score is None: continue
        batch.append({'text':str(text),'label':int(float(score)>=THRESHOLD)})
        if len(batch)>=5000:
            table=pa.Table.from_pylist(batch)
            if writer is None: writer=pq.ParquetWriter(str(tmp),table.schema,compression='snappy')
            writer.write_table(table); count+=len(batch); batch=[]
    if batch:
        table=pa.Table.from_pylist(batch)
        if writer is None: writer=pq.ParquetWriter(str(tmp),table.schema,compression='snappy')
        writer.write_table(table); count+=len(batch)
    if writer: writer.close()
    if not count: raise RuntimeError('No rows cached: '+name)
    tmp.replace(path); print(name,count); return path
splits={s:load_dataset('parquet',data_files=str(cache_split(s)),split='train') for s in ['train','validation','test']}
for k,v in splits.items(): print(k,len(v))

## 2. Tokenization, model, training and evaluation

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
class CommentDataset(Dataset):
    def __init__(self,ds): self.ds=ds
    def __len__(self): return len(self.ds)
    def __getitem__(self,i):
        r=self.ds[i]; x=tokenizer(r['text'],truncation=True,padding='max_length',max_length=MAX_LENGTH,return_tensors='pt')
        return {'input_ids':x['input_ids'][0],'attention_mask':x['attention_mask'][0],'labels':torch.tensor(r['label'],dtype=torch.long)}
def make_loader(ds,bs,shuffle=False): return DataLoader(CommentDataset(ds),batch_size=bs,shuffle=shuffle,num_workers=2 if DEVICE.type=='cuda' else 0)
class Model(nn.Module):
    def __init__(self):
        super().__init__(); self.bert=AutoModel.from_pretrained(MODEL_NAME); self.drop=nn.Dropout(.2); self.head=nn.Linear(self.bert.config.hidden_size,2)
    def forward(self,ids,mask): return self.head(self.drop(self.bert(input_ids=ids,attention_mask=mask).last_hidden_state[:,0]))
def score(y,p):
    pr,re,f,_=precision_recall_fscore_support(y,p,average='binary',zero_division=0)
    return {'accuracy':accuracy_score(y,p),'precision':pr,'recall':re,'f1':f}
@torch.no_grad()
def evaluate(model,dl):
    model.eval(); y=[]; p=[]; prob=[]
    for b in dl:
        q=torch.softmax(model(b['input_ids'].to(DEVICE),b['attention_mask'].to(DEVICE)),1)[:,1]
        y.extend(b['labels'].tolist()); p.extend((q>=.5).long().cpu().tolist()); prob.extend(q.cpu().tolist())
    return score(np.array(y),np.array(p)),np.array(y),np.array(p),np.array(prob)
def train(model,tr,va,epochs=2,lr=2e-5,frozen=False):
    model.to(DEVICE)
    for p in model.bert.parameters(): p.requires_grad=not frozen
    if frozen: groups=[{'params':model.head.parameters(),'lr':lr}]
    else:
        last=model.bert.encoder.layer[-1]; ids={id(p) for p in last.parameters()}
        early=[p for p in model.bert.parameters() if id(p) not in ids]
        groups=[{'params':early,'lr':lr},{'params':last.parameters(),'lr':lr*2},{'params':model.head.parameters(),'lr':lr*5}]
    opt=torch.optim.AdamW(groups,weight_decay=.01); steps=len(tr)*epochs
    sch=get_linear_schedule_with_warmup(opt,max(1,int(.1*steps)),steps); lossfn=nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train(); total=0
        for b in tr:
            opt.zero_grad(); logits=model(b['input_ids'].to(DEVICE),b['attention_mask'].to(DEVICE)); loss=lossfn(logits,b['labels'].to(DEVICE)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sch.step(); total+=loss.item()
        print('epoch',ep+1,'loss',total/max(1,len(tr)),'val',evaluate(model,va)[0])
    return model

## 3. Frozen baseline and initial end-to-end fine-tuning

In [ ]:
train_ds=splits['train']; valid_ds=splits['validation']; test_ds=splits['test']
base_tr=make_loader(train_ds,16,True); base_va=make_loader(valid_ds,16)
baseline=Model(); train(baseline,base_tr,base_va,epochs=2,lr=1e-3,frozen=True)
print('Frozen validation:',evaluate(baseline,base_va)[0])
ft=Model(); train(ft,base_tr,base_va,epochs=2,lr=2e-5,frozen=False)
print('Fine-tuned validation:',evaluate(ft,base_va)[0])

## 4. Hyperparameter tuning

Four configurations are evaluated using validation F1. Test data is not used to select parameters.

In [ ]:
trials=[{'learning_rate':lr,'batch_size':bs} for lr in [2e-5,3e-5] for bs in [16,32]]; rows=[]
for i,h in enumerate(trials,1):
    print('Trial',i,h); m=Model(); tr=make_loader(train_ds,h['batch_size'],True); va=make_loader(valid_ds,h['batch_size'])
    train(m,tr,va,epochs=2,lr=h['learning_rate']); met=evaluate(m,va)[0]; rows.append({**h,**met})
    del m,tr,va; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
tuning=pd.DataFrame(rows).sort_values('f1',ascending=False); display(tuning)
Path('results').mkdir(exist_ok=True); tuning.to_csv('results/hyperparameter_tuning.csv',index=False)
plt.figure(figsize=(8,4)); plt.bar([f"lr={r.learning_rate:g}, bs={r.batch_size}" for r in tuning.itertuples()],tuning.f1); plt.ylabel('Validation F1'); plt.title('Hyperparameter tuning'); plt.xticks(rotation=20); plt.tight_layout(); plt.savefig('results/hyperparameter_tuning.png',dpi=180); plt.show()
best=tuning.iloc[0].to_dict(); best_lr=float(best['learning_rate']); best_bs=int(best['batch_size']); print('Best:',best)

## 5. Final tuned model and held-out test evaluation

In [ ]:
final=Model(); train(final,make_loader(train_ds,best_bs,True),make_loader(valid_ds,best_bs),epochs=2,lr=best_lr)
test_loader=make_loader(test_ds,best_bs); rows=[]
for name,m in [('Frozen BERT',baseline),('Initial fine-tuned BERT',ft),('Tuned fine-tuned BERT',final)]:
    met,y_test,pred,prob=evaluate(m,test_loader); print(name,met); rows.append({'model':name,**met})
print(classification_report(y_test,pred,target_names=['Not toxic','Toxic'],zero_division=0)); print(confusion_matrix(y_test,pred))
pd.DataFrame(rows).to_csv('results/model_comparison.csv',index=False)

## 6. Real-world text predictions

In [ ]:
examples=['Thank you for taking the time to explain your perspective. I respectfully disagree.','You are an idiot and nobody should listen to anything you say.','I think this proposal is poorly designed, and the evidence does not support it.']
def predict(texts):
    final.eval(); x=tokenizer(texts,truncation=True,padding=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
    with torch.no_grad(): q=torch.softmax(final(x['input_ids'],x['attention_mask']),1)[:,1].cpu().tolist()
    return [{'text':t,'toxicity_probability':float(v),'predicted_class':'Toxic' if v>=.5 else 'Not toxic'} for t,v in zip(texts,q)]
preds=predict(examples); display(pd.DataFrame(preds)); pd.DataFrame(preds).to_csv('results/real_world_predictions.csv',index=False)

## 7. SHAP interpretability

SHAP explanations are local token contributions, not causal proof. This step can be computationally expensive.

In [ ]:
import shap
SHAP_DIR=Path('results/shap'); SHAP_DIR.mkdir(parents=True,exist_ok=True)
def shap_predict(texts):
    x=tokenizer(list(texts),truncation=True,padding=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
    with torch.no_grad(): return torch.softmax(final(x['input_ids'],x['attention_mask']),1).cpu().numpy()
explainer=shap.Explainer(shap_predict,shap.maskers.Text(tokenizer),output_names=['Not toxic','Toxic'])
explain_texts=examples+["I appreciate your effort, but I think the explanation is incomplete."]
sv=explainer(explain_texts)
for i in range(len(explain_texts)):
    shap.plots.text(sv[i,:,1]); toks=list(sv.data[i]); vals=np.asarray(sv.values[i,:,1]).reshape(-1); n=min(len(toks),len(vals))
    pd.DataFrame({'token':toks[:n],'shap_value_toxic':vals[:n]}).to_csv(SHAP_DIR/f'example_{i+1}_token_attributions.csv',index=False)

## 8. Save experiment summary

In [ ]:
summary={'model':MODEL_NAME,'threshold':THRESHOLD,'best_hyperparameters':{'learning_rate':best_lr,'batch_size':best_bs},'test_metrics':rows}
with open('results/experiment_summary.json','w') as f: json.dump(summary,f,indent=2)
torch.save(final.state_dict(),'results/final_model_state_dict.pt'); print(json.dumps(summary,indent=2))